# Занятие 2. SQL SELECT и фильтрация данных

## Цель занятия

На прошлом занятии мы создали первую базу данных, таблицу и добавили записи.

Сегодня учимся получать из базы **не все данные подряд**, а только нужные.

К концу занятия студент должен уметь:

- выполнять `SELECT`;
- выбирать отдельные столбцы;
- использовать `WHERE`;
- применять условия `>`, `<`, `=`;
- использовать `AND` и `OR`;
- сортировать данные через `ORDER BY`;
- ограничивать результат через `LIMIT`.

---

## Зачем это нужно в реальной жизни

В реальных приложениях редко нужно показывать все данные сразу.

Примеры:

- интернет-магазин показывает товары дешевле 10 000;
- HR-система ищет кандидатов из конкретного города;
- приложение питания ищет продукты с калориями меньше 300;
- спортивное приложение показывает тренировки длиннее 40 минут;
- учебная система показывает студентов старше 18 лет.

Для этого нужны SQL-запросы с фильтрацией.

---

## Что такое SELECT

`SELECT` — команда SQL для получения данных из таблицы.

Пример:

```sql
SELECT * FROM products;
```

Означает: получить все столбцы и все строки из таблицы `products`.

---

## Что такое WHERE

`WHERE` задаёт условие отбора.

Пример:

```sql
SELECT * FROM products WHERE price > 10000;
```

Означает: получить только товары дороже 10000.

---

## Что такое ORDER BY

`ORDER BY` сортирует данные.

Пример:

```sql
SELECT * FROM products ORDER BY price DESC;
```

Означает: отсортировать товары по цене от дорогих к дешёвым.

---

## Что такое LIMIT

`LIMIT` ограничивает количество строк.

Пример:

```sql
SELECT * FROM products LIMIT 3;
```

Означает: показать только первые 3 строки.

---

## Зачем это нужно в дипломе

В дипломном проекте студенту почти всегда нужно будет:

- искать записи;
- фильтровать данные;
- показывать топ результатов;
- делать простую аналитику;
- выбирать данные под запрос пользователя.

Например:

- HR: кандидаты по городу, опыту, навыкам;
- питание: продукты по калориям и категории;
- спорт: тренировки по цели и длительности;
- магазин: товары по цене и категории.

---

## Связь с GitHub

На этом занятии в проект добавляется логика запросов.

Рекомендуемая ветка:

```bash
git checkout main
git pull origin main
git checkout -b feature/lesson-02-select-filtering
```

Рекомендуемый commit:

```bash
git add .
git commit -m "feat: add select and filtering queries"
git push -u origin feature/lesson-02-select-filtering
```

## Ячейка 1. Подключаем sqlite3 и создаём базу

Создадим базу `shop_lesson02.db`.

В этом занятии будем работать с таблицей товаров.

In [ ]:
import sqlite3

connection = sqlite3.connect("shop_lesson02.db")
cursor = connection.cursor()

print("База данных подключена")

## Ячейка 2. Создаём таблицу products

Таблица будет хранить товары:

- `id` — номер;
- `title` — название;
- `category` — категория;
- `price` — цена;
- `count` — количество на складе.

In [ ]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS products (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT,
    category TEXT,
    price INTEGER,
    count INTEGER
)
""")

connection.commit()

print("Таблица products создана")

## Ячейка 3. Очищаем таблицу и добавляем тестовые данные

`DELETE FROM products` очищает таблицу, чтобы при повторном запуске ноутбука данные не дублировались.

In [ ]:
cursor.execute("DELETE FROM products")

products = [
    ("Ноутбук", "Техника", 70000, 5),
    ("Мышь", "Техника", 1500, 30),
    ("Клавиатура", "Техника", 3500, 20),
    ("Книга Python", "Книги", 2500, 12),
    ("Стол", "Мебель", 12000, 4),
    ("Кресло", "Мебель", 18000, 3),
    ("Блокнот", "Канцелярия", 300, 50),
    ("Ручка", "Канцелярия", 100, 100)
]

cursor.executemany(
    "INSERT INTO products (title, category, price, count) VALUES (?, ?, ?, ?)",
    products
)

connection.commit()

print("Тестовые товары добавлены")

## Ячейка 4. SELECT * — получить все данные

`*` означает: показать все столбцы.

In [ ]:
cursor.execute("SELECT * FROM products")

all_products = cursor.fetchall()

for product in all_products:
    print(product)

assert len(all_products) == 8

## Ячейка 5. SELECT отдельных столбцов

Можно получать не всю таблицу, а только нужные столбцы.

Например: название и цену.

In [ ]:
cursor.execute("SELECT title, price FROM products")

titles_and_prices = cursor.fetchall()

for row in titles_and_prices:
    print(row)

assert len(titles_and_prices) == 8

## Ячейка 6. WHERE: фильтр по цене

Получим товары дороже 10000.

In [ ]:
cursor.execute(
    "SELECT * FROM products WHERE price > ?",
    (10000,)
)

expensive_products = cursor.fetchall()

for product in expensive_products:
    print(product)

assert len(expensive_products) >= 1

## Ячейка 7. WHERE: фильтр по категории

Получим только товары из категории `Техника`.

In [ ]:
cursor.execute(
    "SELECT * FROM products WHERE category = ?",
    ("Техника",)
)

tech_products = cursor.fetchall()

for product in tech_products:
    print(product)

assert len(tech_products) == 3

## Ячейка 8. AND и OR

`AND` означает: должны выполниться оба условия.

`OR` означает: достаточно одного условия.

Сначала найдём технику дороже 3000.

In [ ]:
cursor.execute(
    "SELECT * FROM products WHERE category = ? AND price > ?",
    ("Техника", 3000)
)

filtered_and = cursor.fetchall()

print("Техника дороже 3000:")
for product in filtered_and:
    print(product)

cursor.execute(
    "SELECT * FROM products WHERE category = ? OR category = ?",
    ("Книги", "Канцелярия")
)

filtered_or = cursor.fetchall()

print("\nКниги или канцелярия:")
for product in filtered_or:
    print(product)

assert len(filtered_and) >= 1
assert len(filtered_or) >= 1

## Ячейка 9. ORDER BY и LIMIT

`ORDER BY price DESC` — сортировка по цене от большей к меньшей.

`LIMIT 3` — показать только 3 строки.

In [ ]:
cursor.execute(
    "SELECT * FROM products ORDER BY price DESC LIMIT 3"
)

top_expensive = cursor.fetchall()

for product in top_expensive:
    print(product)

assert len(top_expensive) == 3

## Ячейка 10. Закрываем соединение

После работы соединение с базой нужно закрывать.

In [ ]:
connection.close()

print("Соединение закрыто")

# Итог занятия 2

Сегодня вы научились:

- получать все данные через `SELECT *`;
- выбирать отдельные столбцы;
- фильтровать через `WHERE`;
- использовать `AND` и `OR`;
- сортировать через `ORDER BY`;
- ограничивать результат через `LIMIT`.

## Что коммитить в GitHub

- solved-ноутбук;
- TODO-ноутбук;
- файл базы данных, если преподаватель просит;
- будущий модуль `src/db/queries.py`.

```bash
git add .
git commit -m "feat: add select and filtering queries"
git push -u origin feature/lesson-02-select-filtering
```